# Lab 4 – Fine-Tuning an LLM using LoRA
PUSL3205 – Large Language Models and Generative AI

## Task 1 – Install Required Libraries

In [1]:
%pip install -U transformers datasets peft trl accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -U transformers datasets peft trl accelerate

Note: you may need to restart the kernel to use updated packages.


## Task 2 – Check GPU Availability

In [3]:
import torch, platform, subprocess

print("PyTorch version:", torch.__version__)
print("CUDA (NVIDIA) GPU available:", torch.cuda.is_available())
print("MPS (Apple GPU) available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
elif torch.backends.mps.is_available():
    device = "mps"
    chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip()
    mem = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout) / 1e9
    print("GPU: Apple Silicon GPU –", chip)
    print(f"Unified memory (shared by CPU and GPU): {mem:.0f} GB")
else:
    device = "cpu"
    print("No GPU found – using CPU")

print("Using device:", device)

PyTorch version: 2.15.0
CUDA (NVIDIA) GPU available: False
MPS (Apple GPU) available: True
GPU: Apple Silicon GPU – Apple M4
Unified memory (shared by CPU and GPU): 17 GB
Using device: mps


## Task 3 – Choose the Model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).to(device)

print(f"Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

W1125 17:35:52.704000 47380 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model parameters: 494,032,768


In [5]:
def ask(model, question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask(base_model, "What are the opening hours of TechZone Computers?"))

I'm sorry, but I don't have access to real-time information about specific businesses or their operating hours. The opening and closing times for tech companies can vary depending on factors such as location, business hours, and any seasonal changes. It's best to check directly with TechZone Computers or their official website for the most accurate and up-to-date information regarding their operational hours.


**Before fine-tuning:** The original model does not know anything about TechZone Computers, because it is a fictional shop that was not in its training data. It gives a generic answer saying it has no information about the business.

## Task 4 – Create the Dataset

In [6]:
qa_pairs = [
    ("What are your opening hours?", "TechZone Computers is open from 9:00 AM to 7:00 PM, Monday to Saturday. We are closed on Sundays and public holidays."),
    ("Are you open on Sundays?", "No, TechZone Computers is closed on Sundays and public holidays. We are open Monday to Saturday, 9:00 AM to 7:00 PM."),
    ("Where is TechZone Computers located?", "We are located at No. 45, Galle Road, Colombo 03, on the ground floor of the Galle Road Tech Centre."),
    ("What is your phone number?", "You can call TechZone Computers on +94 11 234 5678 during our opening hours."),
    ("Do you have a WhatsApp number?", "Yes, you can message us on WhatsApp at +94 77 123 4567."),
    ("What is your website?", "Our website is www.techzonecomputers.lk, where you can browse products and check prices."),
    ("Who founded TechZone Computers?", "TechZone Computers was founded in 2015 by Nimal Perera."),
    ("Do you repair laptops?", "Yes, we repair all major laptop brands. Most repairs are completed within 2 to 3 working days."),
    ("How much does a repair diagnosis cost?", "Diagnosis is free at TechZone Computers. We only charge if you approve the repair."),
    ("How long does a laptop screen replacement take?", "If the screen is in stock, we can replace it on the same day. Otherwise it takes 3 to 5 working days."),
    ("How much does a laptop battery replacement cost?", "Laptop battery replacements start from LKR 8,000, depending on the model."),
    ("Can you remove viruses from my computer?", "Yes, our virus removal service costs LKR 2,500 and is usually completed on the same day."),
    ("Do you install Windows?", "Yes, Windows installation with all drivers costs LKR 3,000. A genuine Windows licence is sold separately."),
    ("Do you offer data recovery?", "Yes, we offer data recovery starting from LKR 5,000, depending on the damage to the drive."),
    ("Do you repair MacBooks?", "Yes, we repair MacBooks. We have an Apple-trained technician on our team."),
    ("Do you repair mobile phones?", "No, TechZone Computers specialises in computers, laptops and printers only. We do not repair mobile phones."),
    ("Do you offer laptop cleaning?", "Yes, our internal cleaning and thermal paste replacement service costs LKR 3,500 and helps stop overheating."),
    ("Can you upgrade my RAM or SSD?", "Yes, and installation is free when you buy the RAM or SSD from TechZone Computers."),
    ("What warranty do you give on repairs?", "All repairs come with a 3-month warranty on both parts and labour."),
    ("What warranty comes with a new laptop?", "New laptops come with a 1-year manufacturer warranty. You can add a 2-year TechZone extended warranty for LKR 15,000."),
    ("Which laptop brands do you sell?", "We sell Dell, HP, Lenovo, ASUS, Acer and Apple MacBook laptops."),
    ("What is the cheapest laptop you sell?", "Our entry-level laptops start from LKR 150,000."),
    ("Which laptop do you recommend for students?", "For students we recommend the Lenovo IdeaPad or HP 15 series. They are affordable and good for studying and assignments."),
    ("Do you sell refurbished laptops?", "Yes, we sell refurbished laptops, and they come with a 6-month TechZone warranty."),
    ("Can you build a custom gaming PC?", "Yes, we build custom gaming PCs within 5 working days, and assembly is free."),
    ("Do you sell printers and ink?", "Yes, we sell HP, Canon and Epson printers along with original ink cartridges."),
    ("What accessories do you sell?", "We sell keyboards, mice, headsets, laptop bags, monitors, cables and external hard drives."),
    ("Do you give student discounts?", "Yes, students get 10% off laptops and accessories when they show a valid student ID."),
    ("What payment methods do you accept?", "We accept cash, Visa and Mastercard, and bank transfers."),
    ("Do you have installment plans?", "Yes, we offer 0% installment plans for up to 12 months with selected bank credit cards."),
    ("Do you deliver?", "Yes, delivery is free within Colombo for orders above LKR 50,000. Island-wide delivery takes 3 working days and costs LKR 1,500."),
    ("What is your return policy?", "You can return unopened products within 7 days with the original receipt."),
    ("Can I trade in my old laptop?", "Yes, you can trade in your old laptop for credit towards a new one. We assess its value in store."),
    ("Do you price match?", "Yes, we match the price of any authorised retailer in Colombo for the same item in stock."),
    ("Do you have a loyalty program?", "Yes, TechZone Rewards gives you 1 point for every LKR 1,000 spent. 100 points can be exchanged for a LKR 1,000 voucher."),
    ("Can I pre-order a laptop?", "Yes, you can pre-order with a 20% deposit, and we will contact you when it arrives."),
    ("How can I check the status of my repair?", "Call us or send a WhatsApp message with your job number and we will give you an update."),
    ("Do you provide on-site support for businesses?", "Yes, on-site support within Colombo costs LKR 5,000 per visit. We also offer annual maintenance contracts for businesses."),
    ("Is there parking at the shop?", "Yes, there is free customer parking for 1 hour behind the shop."),
    ("Do you run any workshops?", "Yes, we run a free PC Basics workshop on the first Saturday of every month at 10:00 AM."),
]

print("Number of Q&A pairs:", len(qa_pairs))

Number of Q&A pairs: 40


In [7]:
from datasets import Dataset
import json

train_data = [
    {"messages": [
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]}
    for q, a in qa_pairs
]
train_dataset = Dataset.from_list(train_data)

# save a copy of the dataset as a file (to include on GitHub)
with open("computer_shop_qa.json", "w") as f:
    json.dump([{"question": q, "answer": a} for q, a in qa_pairs], f, indent=2)

print(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['messages'],
    num_rows: 40
})
{'messages': [{'role': 'user', 'content': 'What are your opening hours?'}, {'role': 'assistant', 'content': 'TechZone Computers is open from 9:00 AM to 7:00 PM, Monday to Saturday. We are closed on Sundays and public holidays.'}]}


In [8]:
print(tokenizer.apply_chat_template(train_data[0]["messages"], tokenize=False))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What are your opening hours?<|im_end|>
<|im_start|>assistant
TechZone Computers is open from 9:00 AM to 7:00 PM, Monday to Saturday. We are closed on Sundays and public holidays.<|im_end|>



**Explanation:** Each example is stored as a chat with a user question and the assistant's answer. The tokenizer's chat template adds special tokens (<|im_start|>, <|im_end|>) to mark who is speaking, plus Qwen's default system message. The model is fine-tuned to produce the assistant's answer in this format.

## Task 5 – Configure LoRA

In [9]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,                 # rank: size of the small LoRA matrices
    lora_alpha=32,        # scaling factor for the LoRA update
    lora_dropout=0.05,    # dropout to reduce overfitting
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",      # attention layers
                    "gate_proj", "up_proj", "down_proj"],        # feed-forward layers
    task_type="CAUSAL_LM",
)
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.22.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'up_proj', 'o_proj', 'gate_proj', 'q_proj', 'down_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, astra_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=Non

## Task 6 – Supervised Fine-Tuning with TRL's SFTTrainer

In [10]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="lora-techzone",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_length=256,
    save_strategy="no",
    report_to="none",
    bf16=False,                     
    dataloader_pin_memory=False,    

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

SyntaxError: '(' was never closed (123091728.py, line 3)

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

logs = [l for l in trainer.state.log_history if "loss" in l]
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs], marker="o")
plt.xlabel("Training step")
plt.ylabel("Training loss")
plt.title("LoRA Fine-Tuning Loss")
plt.grid(True, alpha=0.3)
plt.show()

**Observation:** The training loss dropped quickly from about 1.8 to about 0.07 over 100 steps (10 epochs). This shows the LoRA adapter learned the shop's answers very well. Such a low loss on a small dataset also means the model may partly memorise the exact answers, so it is important to test it with new questions that were not in the training data.

## Task 7 & 8 – Compare the Original and Fine-Tuned Models

In [ ]:
test_questions = [
    
    ("Seen", "What are your opening hours?"),
    ("Seen", "Do you repair mobile phones?"),
   
    ("Unseen", "What time does TechZone close on Saturday?"),
    ("Unseen", "I'm a university student. Can I get a discount on a laptop?"),
    ("Unseen", "My laptop has a virus. Can you fix it and how much will it cost?"),
    ("Unseen", "Can I pay for a laptop in monthly payments?"),
    ("Unseen", "Can you fix my cracked iPhone screen?"),
    
    ("Not in dataset", "Do you sell drones?"),
   
    ("General", "What is the difference between an SSD and an HDD?"),
]

In [ ]:
ft_model = trainer.model
ft_model.eval()

results = []
for kind, q in test_questions:
    with ft_model.disable_adapter():          
        original = ask(ft_model, q)
    finetuned = ask(ft_model, q)              
    results.append({"Type": kind, "Question": q, "Original model": original, "Fine-tuned model": finetuned})

    print(f"[{kind}] Q: {q}")
    print(f"  ORIGINAL:   {original}")
    print(f"  FINE-TUNED: {finetuned}")
    print("-" * 100)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
results_df = pd.DataFrame(results)
results_df

In [ ]:
trainer.save_model("techzone-lora-adapter")
tokenizer.save_pretrained("techzone-lora-adapter")

import os
size_mb = sum(os.path.getsize(os.path.join("techzone-lora-adapter", f)) for f in os.listdir("techzone-lora-adapter")) / 1e6
print(f"Adapter saved – total size: {size_mb:.1f} MB (the full model is ~2,000 MB)")